# Data Loading and Extraction

In [ ]:
import os
import json
import glob
import pandas as pd

# Path to the JSON files in the Kaggle environment
file_pattern = 'ipl-2008-2026/*.json'
json_files = glob.glob(file_pattern)

processed_matches = []

for file in json_files:
    with open(file, 'r') as f:
        try:
            data = json.load(f)
        except json.JSONDecodeError:
            continue
        
        info = data.get('info', {})
        
        # 1. Match/Event Metadata
        city = info.get('city', 'Unknown')
        dates = info.get('dates', [None])[0]
        event = info.get('event', {})
        match_number = event.get('match_number', 'N/A')
        event_name = event.get('name', 'Unknown')
        season = info.get('season', 'Unknown')
        venue = info.get('venue', 'Unknown')
        match_type = info.get('match_type', 'Unknown')
        gender = info.get('gender', 'Unknown')
        team_type = info.get('team_type', 'Unknown')
        balls_per_over = info.get('balls_per_over', 6)
        overs_limit = info.get('overs', 20)
        
        # 2. Teams & Players
        teams = info.get('teams', [])
        team1 = teams[0] if len(teams) > 0 else 'Unknown'
        team2 = teams[1] if len(teams) > 1 else 'Unknown'
        
        players_dict = info.get('players', {})
        team1_players = ", ".join(players_dict.get(team1, []))
        team2_players = ", ".join(players_dict.get(team2, []))
        
        # 3. Toss
        toss = info.get('toss', {})
        toss_winner = toss.get('winner', 'Unknown')
        toss_decision = toss.get('decision', 'Unknown')
        
        # 4. Outcome & Player of Match
        outcome = info.get('outcome', {})
        winner = outcome.get('winner', 'None')
        result_type = outcome.get('result', 'complete') # Captures 'tie' or 'no result'
        win_by_runs = outcome.get('by', {}).get('runs', 0)
        win_by_wickets = outcome.get('by', {}).get('wickets', 0)
        
        player_of_match_list = info.get('player_of_match', [])
        player_of_match = ", ".join(player_of_match_list) if player_of_match_list else 'None'
        
        # 5. Officials
        officials = info.get('officials', {})
        match_referee = ", ".join(officials.get('match_referees', ['Unknown']))
        reserve_umpire = ", ".join(officials.get('reserve_umpires', ['Unknown']))
        tv_umpire = ", ".join(officials.get('tv_umpires', ['Unknown']))
        
        umpires = officials.get('umpires', ['Unknown', 'Unknown'])
        umpire1 = umpires[0] if len(umpires) > 0 else 'Unknown'
        umpire2 = umpires[1] if len(umpires) > 1 else 'Unknown'

        # 6. Extract Runs and Wickets from Innings (Aggregated, not ball-by-ball)
        team_stats = {team1: {'runs': 0, 'wickets': 0}, team2: {'runs': 0, 'wickets': 0}}
        innings_data = data.get('innings', [])
        
        for inning in innings_data:
            batting_team = inning.get('team')
            if batting_team not in team_stats:
                continue 
                
            for over in inning.get('overs', []):
                for delivery in over.get('deliveries', []):
                    team_stats[batting_team]['runs'] += delivery.get('runs', {}).get('total', 0)
                    if 'wickets' in delivery:
                        team_stats[batting_team]['wickets'] += len(delivery['wickets'])

        # 7. Compile the exhaustive match record
        match_record = {
            'event_name': event_name,
            'season': season,
            'match_number': match_number,
            'date': dates,
            'city': city,
            'venue': venue,
            'team1': team1,
            'team2': team2,
            'toss_winner': toss_winner,
            'toss_decision': toss_decision,
            'team1_runs': team_stats[team1]['runs'],
            'team1_wickets': team_stats[team1]['wickets'],
            'team2_runs': team_stats[team2]['runs'],
            'team2_wickets': team_stats[team2]['wickets'],
            'winner': winner,
            'result_type': result_type,
            'win_by_runs': win_by_runs,
            'win_by_wickets': win_by_wickets,
            'player_of_match': player_of_match,
            'match_referee': match_referee,
            'umpire1': umpire1,
            'umpire2': umpire2,
            'tv_umpire': tv_umpire,
            'reserve_umpire': reserve_umpire,
            'match_type': match_type,
            'overs_limit': overs_limit,
            'balls_per_over': balls_per_over,
            'gender': gender,
            'team_type': team_type,
            'team1_players': team1_players,
            'team2_players': team2_players
        }
        
        processed_matches.append(match_record)

# 8. Convert to DataFrame, Sort, and Save
df = pd.DataFrame(processed_matches)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(by=['date', 'match_number']).reset_index(drop=True)

output_path = 'ipl_comprehensive_dataset.csv'
df.to_csv(output_path, index=False)
print(f"Successfully compiled {len(df)} matches with extended fields. File saved to {output_path}")

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

# Year-wise Data Extraction

In [3]:
import os
import json
import glob
import pandas as pd

# Path to the JSON files in the Kaggle environment
file_pattern = 'ipl-2008-2026/*.json'
json_files = glob.glob(file_pattern)

processed_matches = []

for file in json_files:
    with open(file, 'r') as f:
        try:
            data = json.load(f)
        except json.JSONDecodeError:
            continue
        
        info = data.get('info', {})
        
        # 1. Match/Event Metadata
        city = info.get('city', 'Unknown')
        dates = info.get('dates', [None])[0]
        season = info.get('season', 'Unknown')
        venue = info.get('venue', 'Unknown')
        overs_limit = info.get('overs', 20)
        
        # 2. Teams & Players
        teams = info.get('teams', [])
        team1 = teams[0] if len(teams) > 0 else 'Unknown'
        team2 = teams[1] if len(teams) > 1 else 'Unknown'
        
        players_dict = info.get('players', {})
        team1_players = ", ".join(players_dict.get(team1, []))
        team2_players = ", ".join(players_dict.get(team2, []))
        
        # 3. Toss
        toss = info.get('toss', {})
        toss_winner = toss.get('winner', 'Unknown')
        toss_decision = toss.get('decision', 'Unknown')
        
        # 4. Outcome & Player of Match
        outcome = info.get('outcome', {})
        winner = outcome.get('winner', 'None')
        result_type = outcome.get('result', 'complete')
        win_by_runs = outcome.get('by', {}).get('runs', 0)
        win_by_wickets = outcome.get('by', {}).get('wickets', 0)
        
        player_of_match_list = info.get('player_of_match', [])
        player_of_match = ", ".join(player_of_match_list) if player_of_match_list else 'None'
        
        # 5. Officials
        officials = info.get('officials', {})
        match_referee = ", ".join(officials.get('match_referees', ['Unknown']))
        reserve_umpire = ", ".join(officials.get('reserve_umpires', ['Unknown']))
        tv_umpire = ", ".join(officials.get('tv_umpires', ['Unknown']))
        
        umpires = officials.get('umpires', ['Unknown', 'Unknown'])
        umpire1 = umpires[0] if len(umpires) > 0 else 'Unknown'
        umpire2 = umpires[1] if len(umpires) > 1 else 'Unknown'

        # 6. Extract Runs and Wickets from Innings
        team_stats = {team1: {'runs': 0, 'wickets': 0}, team2: {'runs': 0, 'wickets': 0}}
        innings_data = data.get('innings', [])
        
        for inning in innings_data:
            batting_team = inning.get('team')
            if batting_team not in team_stats:
                continue 
                
            for over in inning.get('overs', []):
                for delivery in over.get('deliveries', []):
                    team_stats[batting_team]['runs'] += delivery.get('runs', {}).get('total', 0)
                    if 'wickets' in delivery:
                        team_stats[batting_team]['wickets'] += len(delivery['wickets'])

        # 7. Compile the match record (Excluded requested fields)
        match_record = {
            'date': dates,
            'season': season,
            'city': city,
            'venue': venue,
            'team1': team1,
            'team2': team2,
            'toss_winner': toss_winner,
            'toss_decision': toss_decision,
            'team1_runs': team_stats[team1]['runs'],
            'team1_wickets': team_stats[team1]['wickets'],
            'team2_runs': team_stats[team2]['runs'],
            'team2_wickets': team_stats[team2]['wickets'],
            'winner': winner,
            'result_type': result_type,
            'win_by_runs': win_by_runs,
            'win_by_wickets': win_by_wickets,
            'player_of_match': player_of_match,
            'match_referee': match_referee,
            'umpire1': umpire1,
            'umpire2': umpire2,
            'tv_umpire': tv_umpire,
            'reserve_umpire': reserve_umpire,
            'overs_limit': overs_limit,
            'team1_players': team1_players,
            'team2_players': team2_players
        }
        
        processed_matches.append(match_record)

# 8. Convert to DataFrame
df = pd.DataFrame(processed_matches)

# Clean and parse the date, extract the year for grouping
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year

# 9. Sort by Date and Set it as the Index
df = df.sort_values(by='date')
df = df.set_index('date')

# 10. Split and Save into Year-wise CSV Files
output_dir = './dataset/'

df.to_csv(f'{output_dir}ipl_data.csv', index=True)

# Group by the extracted year
for year, group_df in df.groupby('year'):
    # Drop the temporary 'year' column to keep the dataset perfectly clean
    clean_group_df = group_df.drop(columns=['year'])
    
    # Save each year as a separate CSV
    file_path = os.path.join(output_dir, f'ipl_matches_{year}.csv')
    
    # We leave index=True here because the index is now the 'date' column
    clean_group_df.to_csv(file_path, index=True) 
    print(f"Saved: {file_path} ({len(clean_group_df)} matches)")

print("All year-wise files have been generated successfully!")

Saved: ./dataset/ipl_matches_2008.csv (58 matches)
Saved: ./dataset/ipl_matches_2009.csv (57 matches)
Saved: ./dataset/ipl_matches_2010.csv (60 matches)
Saved: ./dataset/ipl_matches_2011.csv (73 matches)
Saved: ./dataset/ipl_matches_2012.csv (74 matches)
Saved: ./dataset/ipl_matches_2013.csv (76 matches)
Saved: ./dataset/ipl_matches_2014.csv (60 matches)
Saved: ./dataset/ipl_matches_2015.csv (59 matches)
Saved: ./dataset/ipl_matches_2016.csv (60 matches)
Saved: ./dataset/ipl_matches_2017.csv (59 matches)
Saved: ./dataset/ipl_matches_2018.csv (60 matches)
Saved: ./dataset/ipl_matches_2019.csv (60 matches)
Saved: ./dataset/ipl_matches_2020.csv (60 matches)
Saved: ./dataset/ipl_matches_2021.csv (60 matches)
Saved: ./dataset/ipl_matches_2022.csv (74 matches)
Saved: ./dataset/ipl_matches_2023.csv (74 matches)
Saved: ./dataset/ipl_matches_2024.csv (71 matches)
Saved: ./dataset/ipl_matches_2025.csv (74 matches)
Saved: ./dataset/ipl_matches_2026.csv (74 matches)
All year-wise files have been g

In [ ]:
display(df.head())
display(df.tail())
display(df.info())